# PlantMetWiki figures

Generates publication-quality figures by querying the **local RDF bundles** produced by this repository — no Virtuoso required.

**Workflow:**
```
output/bundles/*.ttl  →  rdflib SPARQL  →  figures/output/*.csv  →  figures/output/*.pdf/.svg
```

Reference CSVs from a previous run are in `figures/reference/` for comparison only.

**Kernel:** `plantmetwiki-rdf`

---

## ⚠️ rdflib limitations (same as explore_taxonomy_rdf.ipynb)

| ✅ Fast | ❌ Avoid |
|---|---|
| Simple `SELECT` with 1-2 triple patterns per graph | Merging `g_core + g_tax` — huge combined graph |
| `COUNT` / `GROUP BY` on one variable | `STRSTARTS(?a, STR(?b))` over large sets — O(n²) |
| Queries on taxonomy bundle alone (~4 MB) | Joining two large graphs in SPARQL |

Where queries need **both** graphs (e.g. species per pathway), we run them separately and **join in Python**.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from rdflib import Graph, URIRef

# ── Paths ─────────────────────────────────────────────────────────────────────
BUNDLES   = Path("../output/bundles")
OUT_DIR   = Path("figures/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect latest version from bundle filenames
core_files = sorted(BUNDLES.glob("all-*.ttl"))
assert core_files, f"No core bundle found in {BUNDLES}"
VERSION = core_files[-1].stem.replace("all-", "")
print(f"Version: {VERSION}")

CORE_FILE     = BUNDLES / f"all-{VERSION}.ttl"
TAXONOMY_FILE = BUNDLES / f"all_gpml_taxonomy_extra-{VERSION}.ttl"

plt.rcParams.update({"figure.dpi": 150, "font.size": 10})

PREFIXES = """
PREFIX wp:      <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi:    <http://purl.obolibrary.org/obo/NCBITaxon_>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dcterms: <http://purl.org/dc/terms/>
"""

def sparql(g: Graph, query: str) -> pd.DataFrame:
    results = g.query(PREFIXES + query)
    return pd.DataFrame(results, columns=[str(v) for v in results.vars])

def save_csv(df: pd.DataFrame, name: str) -> pd.DataFrame:
    path = OUT_DIR / name
    df.to_csv(path, index=False)
    print(f"  Saved {len(df):,} rows → {path}")
    return df

def save_fig(fig: plt.Figure, name: str) -> None:
    for ext in ("pdf", "svg"):
        fig.savefig(OUT_DIR / f"{name}.{ext}", bbox_inches="tight")
    print(f"  Saved → {OUT_DIR}/{name}.{{pdf,svg}}")

print(f"Core bundle:     {CORE_FILE.name}  ({CORE_FILE.stat().st_size/1e6:.0f} MB)")
print(f"Taxonomy bundle: {TAXONOMY_FILE.name}  ({TAXONOMY_FILE.stat().st_size/1e6:.1f} MB)")

## Load bundles

Taxonomy (~4 MB, fast). Core (~300 MB, ~1-2 min). Keep them **separate**.

In [ ]:
print("Loading taxonomy extra bundle...")
g_tax = Graph()
g_tax.parse(str(TAXONOMY_FILE), format="turtle")
print(f"  {len(g_tax):,} triples")

In [ ]:
print("Loading core bundle (~1-2 min)...")
g_core = Graph()
g_core.parse(str(CORE_FILE), format="turtle")
print(f"  {len(g_core):,} triples")

---
## 1. Query local bundles → CSVs

Each cell queries `g_core` or `g_tax` and saves a fresh CSV to `figures/output/`.

In [ ]:
# Genes (GeneProduct) per pathway
print("Counting genes per pathway...")
genes = save_csv(sparql(g_core, """
SELECT ?pwID ?title (COUNT(DISTINCT ?node) AS ?count)
WHERE {
    ?node a wp:GeneProduct ; wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
"""), "genes_per_pathway.csv")
genes.head(3)

In [ ]:
# Metabolites per pathway
print("Counting metabolites per pathway...")
metabolites = save_csv(sparql(g_core, """
SELECT ?pwID ?title (COUNT(DISTINCT ?node) AS ?count)
WHERE {
    ?node a wp:Metabolite ; wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
"""), "metabolites_per_pathway.csv")
metabolites.head(3)

In [ ]:
# Enzymes (Protein DataNodes) per pathway
print("Counting enzymes per pathway...")
enzymes = save_csv(sparql(g_core, """
SELECT ?pwID ?title (COUNT(DISTINCT ?node) AS ?count)
WHERE {
    ?node a wp:Protein ; wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
"""), "enzymes_per_pathway.csv")
enzymes.head(3)

In [ ]:
# Conversions per pathway
print("Counting conversions per pathway...")
conversions = save_csv(sparql(g_core, """
SELECT ?pwID ?title (COUNT(DISTINCT ?interaction) AS ?count)
WHERE {
    ?interaction a wp:Conversion ; wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
"""), "conversions_per_pathway.csv")
conversions.head(3)

In [ ]:
# Interaction type counts
print("Counting interaction types...")
int_types = save_csv(sparql(g_core, """
SELECT ?type (COUNT(DISTINCT ?interaction) AS ?n)
WHERE {
    ?interaction a ?type .
    FILTER(STRSTARTS(STR(?type), "http://vocabularies.wikipathways.org/wp#"))
}
GROUP BY ?type
ORDER BY DESC(?n)
"""), "interaction_types.csv")
int_types

In [ ]:
# Species per pathway — two separate queries joined in Python
# (avoids slow cross-graph join in rdflib)
print("Step 1: DataNode → pathway map from core...")
node_to_pathway = sparql(g_core, """
SELECT DISTINCT ?node ?pwID
WHERE {
    ?node wp:isPartOf ?pwID .
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
    FILTER(CONTAINS(STR(?pwID), "/pathways/"))
}
""")
print(f"  {len(node_to_pathway):,} DataNode→pathway pairs")

print("Step 2: DataNode → taxon map from taxonomy-extra...")
node_to_taxon = sparql(g_tax, """
SELECT DISTINCT ?node ?species
WHERE {
    ?node wp:organism ?species .
    FILTER(?species != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
}
""")
print(f"  {len(node_to_taxon):,} DataNode→taxon pairs")

print("Step 3: Python join → species per pathway...")
merged_sp = node_to_pathway.merge(node_to_taxon, on="node", how="inner")
species_pw = (
    merged_sp.groupby("pwID")["species"]
    .nunique()
    .reset_index()
    .rename(columns={"species": "count"})
    .sort_values("count", ascending=False)
)
save_csv(species_pw, "species_per_pathway.csv")
species_pw.head(5)

In [ ]:
# Per-species metrics — also a Python join
print("Computing per-species metrics (Python join)...")

# Annotated DataNodes with their type and pathway
node_types = sparql(g_core, """
SELECT DISTINCT ?node ?type ?pwID
WHERE {
    ?node a ?type ; wp:isPartOf ?pwID .
    FILTER(?type IN (wp:GeneProduct, wp:Protein, wp:Metabolite))
    FILTER(CONTAINS(STR(?pwID), "/pathways/"))
}
""")

joined = node_to_taxon.merge(node_types, on="node", how="inner")

WP = "http://vocabularies.wikipathways.org/wp#"
per_species = (
    joined.groupby("species")
    .agg(
        pathways  = ("pwID",  "nunique"),
        genes     = ("node",  lambda x: x[joined.loc[x.index, "type"] == WP+"GeneProduct"].nunique()),
        enzymes   = ("node",  lambda x: x[joined.loc[x.index, "type"] == WP+"Protein"].nunique()),
        metabolites=("node", lambda x: x[joined.loc[x.index, "type"] == WP+"Metabolite"].nunique()),
    )
    .reset_index()
    .sort_values("pathways", ascending=False)
)
per_species["species"] = per_species["species"].apply(
    lambda x: str(x).split("/")[-1].replace("NCBITaxon_", "ncbi:") if str(x).startswith("http") else x
)
save_csv(per_species, "per_species_nrs.csv")
per_species.head(5)

In [ ]:
# Pathway titles
print("Fetching pathway titles...")
titles = save_csv(sparql(g_core, """
SELECT DISTINCT ?pwID ?title
WHERE {
    ?pwID a wp:Pathway ; rdfs:label ?title .
    FILTER(CONTAINS(STR(?pwID), "/pathways/"))
}
"""), "pathway_titles.csv")
print(f"  {len(titles):,} pathway titles")

---
## 2. Normalize

In [ ]:
def norm(df, count_col, count_name):
    df = df.copy()
    df = df.rename(columns={count_col: count_name})
    df[count_name] = pd.to_numeric(df[count_name], errors="coerce").fillna(0)
    if "pwID" in df.columns:
        df["pwID"] = df["pwID"].astype(str)
    return df

genes       = norm(genes,       "count", "genes")
metabolites = norm(metabolites, "count", "metabolites")
enzymes     = norm(enzymes,     "count", "enzymes")
conversions = norm(conversions, "count", "conversions")
species_pw  = norm(species_pw,  "count", "species")

int_types["n"] = pd.to_numeric(int_types["n"], errors="coerce").fillna(0).astype(int)
WP_INTERACTION = "http://vocabularies.wikipathways.org/wp#Interaction"
total_interactions = int(int_types.loc[int_types["type"] == WP_INTERACTION, "n"].iloc[0])

print("✅ Data ready")
print(f"  Pathways with gene data:       {len(genes):,}")
print(f"  Pathways with metabolite data: {len(metabolites):,}")
print(f"  Total interactions:            {total_interactions:,}")
print(f"  Species in per-species table:  {len(per_species):,}")

---
## 3. Figure: Overview bar chart (log scale)

In [ ]:
overview = {
    "Pathways":     len(genes),
    "Genes":        int(genes["genes"].sum()),
    "Metabolites":  int(metabolites["metabolites"].sum()),
    "Enzymes":      int(enzymes["enzymes"].sum()),
    "Interactions": total_interactions,
}
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(list(overview.keys()), list(overview.values()))
ax.set_yscale("log")
ax.set_ylabel("Count (log scale)")
ax.set_title("PlantMetWiki — content overview")
for bar, val in zip(bars, overview.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.1,
            f"{val:,}", ha="center", va="bottom", fontsize=9)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_overview_barlog")
plt.show()

---
## 4. Figure: Cumulative coverage curves

In [ ]:
def cumulative_curve(df, col):
    s = df[col].sort_values(ascending=False)
    total = s.sum()
    return (s.cumsum()/total).values if total > 0 else s.cumsum().values

curves = {
    "Genes":           cumulative_curve(genes, "genes"),
    "Enzymes":         cumulative_curve(enzymes, "enzymes"),
    "Metabolites":     cumulative_curve(metabolites, "metabolites"),
    "Conversions":     cumulative_curve(conversions, "conversions"),
    "Species/pathway": cumulative_curve(species_pw, "species"),
}
styles = {
    "Genes":           dict(linestyle="-",  marker="o", markevery=100),
    "Enzymes":         dict(linestyle="--", marker="s", markevery=100),
    "Metabolites":     dict(linestyle="-.", marker="^", markevery=100),
    "Conversions":     dict(linestyle=":",  marker="x", markevery=100),
    "Species/pathway": dict(linestyle="--", marker="D", markevery=100),
}
fig, ax = plt.subplots(figsize=(7, 4.8))
for label, curve in curves.items():
    ax.plot(range(1, len(curve)+1), curve, label=label, linewidth=2, **styles[label])
ax.set_xlabel("Pathways (ranked by contribution)")
ax.set_ylabel("Cumulative fraction of total")
ax.set_ylim(0, 1.01)
ax.set_title("Cumulative coverage of PlantMetWiki content")
ax.legend()
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_cumulative_coverage")
plt.show()

---
## 5. Figure: Interaction types (2-panel)

In [ ]:
label_map = {
    "http://vocabularies.wikipathways.org/wp#DirectedInteraction":      "Directed interaction",
    "http://vocabularies.wikipathways.org/wp#Conversion":               "Biochemical conversion",
    "http://vocabularies.wikipathways.org/wp#Catalysis":                "Catalysis",
    "http://vocabularies.wikipathways.org/wp#TranscriptionTranslation": "Transcription/translation",
    "http://vocabularies.wikipathways.org/wp#Inhibition":               "Inhibition",
    "http://vocabularies.wikipathways.org/wp#Stimulation":              "Stimulation",
    "http://vocabularies.wikipathways.org/wp#Binding":                  "Binding",
}
sub = int_types[int_types["type"] != WP_INTERACTION].copy()
sub["label"] = sub["type"].map(label_map).fillna(sub["type"])
sub["pct"]   = 100 * sub["n"] / total_interactions
sub = sub.sort_values("n", ascending=True)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.5, 4.2),
                                gridspec_kw={"width_ratios": [1, 3]})
ax0.bar(["Total\ninteractions"], [total_interactions])
ax0.set_ylabel("Count"); ax0.set_title("A")
ax0.text(0, total_interactions, f"{total_interactions:,}",
         ha="center", va="bottom", fontsize=10)
ax0.spines[["top","right"]].set_visible(False)

ax1.barh(sub["label"], sub["n"])
ax1.set_xlabel("Number of interactions"); ax1.set_title("B")
xmax = sub["n"].max()
for y, (n, pct) in enumerate(zip(sub["n"], sub["pct"])):
    ax1.text(n+xmax*0.01, y, f"{n:,}  ({pct:.1f}%)", va="center", fontsize=9)
ax1.set_xlim(0, xmax*1.25)
ax1.spines[["top","right"]].set_visible(False)

fig.suptitle("Interaction types in PlantMetWiki", y=1.02)
plt.tight_layout()
save_fig(fig, "plantmetwiki_interaction_types")
plt.show()

---
## 6. Figure: Species metrics — stacked bar (top 50)

In [ ]:
top50 = per_species.sort_values("pathways", ascending=False).head(50).copy()
stack_metrics = [c for c in ["genes","enzymes","metabolites"] if c in top50.columns]
COLORS = {"genes": "C1", "enzymes": "C2", "metabolites": "C3"}

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(top50))
bottom = np.zeros(len(top50))
for m in stack_metrics:
    ax.bar(x, top50[m].values, bottom=bottom, label=m.capitalize(), color=COLORS[m])
    bottom += top50[m].values
ax.set_xticks(x)
ax.set_xticklabels(top50["species"], rotation=60, ha="right", fontsize=8)
ax.set_ylabel("Count")
ax.set_title("PlantMetWiki content by species (top 50)")
ax.legend(ncol=3, frameon=False)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_species_metrics_stacked_top50")
plt.show()

---
## 7. Figure: Scatter — genes vs metabolites (sized by species count)

In [ ]:
merged = metabolites[["pwID","metabolites"]].merge(
    genes[["pwID","genes"]], on="pwID", how="outer"
).merge(
    species_pw[["pwID","species"]], on="pwID", how="outer"
).fillna(0)
merged = merged.merge(titles.rename(columns={"pwID":"pwID","title":"title"}),
                      on="pwID", how="left")
merged = merged[(merged["metabolites"] > 0) | (merged["genes"] > 0)].copy()

sp_min, sp_max = merged["species"].min(), merged["species"].max()
size = 6 + (merged["species"] - sp_min) / max(sp_max - sp_min, 1) * 154

bins   = sorted(set([0, 1, 2, 5, 10, int(sp_max)]))
blabels = [f"{bins[i-1]+1 if i>1 else 0}-{bins[i]}" if bins[i-1]+1 != bins[i]
           else str(bins[i]) for i in range(1, len(bins))]
merged["sp_bin"]  = pd.cut(merged["species"], bins=bins, include_lowest=True,
                            right=True, labels=blabels)
merged["sp_code"] = merged["sp_bin"].cat.codes

fig, ax = plt.subplots(figsize=(6.8, 5.4))
ax.scatter(merged["metabolites"], merged["genes"],
           s=size, c=merged["sp_code"], cmap="viridis",
           alpha=0.35, edgecolors="none")
ax.set_xlabel("Metabolites per pathway")
ax.set_ylabel("Genes per pathway")
ax.set_title("Pathway content in PlantMetWiki")

cmap = plt.get_cmap("viridis")
norm_c = mpl.colors.Normalize(vmin=0, vmax=max(1, len(blabels)-1))
handles = [Line2D([0],[0], marker="o", linestyle="None",
                  markerfacecolor=cmap(norm_c(i)), markeredgecolor="none",
                  markersize=8, alpha=0.8) for i in range(len(blabels))]
ax.legend(handles, blabels, title="Species count", loc="best", frameon=False)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "scatter_genes_vs_metabolites_size_species")

merged.sort_values(["genes","metabolites"], ascending=False)[
    ["pwID","title","genes","metabolites","species"]
].head(50).to_csv(OUT_DIR / "top_pathways_by_genes.csv", index=False)
print("Saved top_pathways_by_genes.csv")
plt.show()

---
## Sandbox

In [ ]:
# Add new queries or figures here
# Use g_core for pathway/interaction structure, g_tax for species annotations
pass